In [84]:
pip install langchain


Note: you may need to restart the kernel to use updated packages.


In [85]:
pip install -qU langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [86]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key="Your-API-Key)

response = llm.invoke('The first person to climb Mount Everest')
print(response.content)

The first people to climb Mount Everest were Sir Edmund Hillary from New Zealand and Tenzing Norgay, a Nepali Sherpa mountaineer. They successfully reached the summit of Mount Everest on May 29, 1953, as part of a British expedition led by John Hunt.

Sir Edmund Hillary and Tenzing Norgay used the South Col route to climb the mountain, and they spent about 15 minutes at the summit, taking photographs and collecting samples. Their historic achievement marked the first time humans had reached the highest point on Earth, and it was a major milestone in the history of mountaineering.

Both Hillary and Norgay were hailed as heroes and received international recognition for their achievement. They were knighted by Queen Elizabeth II, and their climb helped to establish Mount Everest as a premier destination for adventure seekers and mountaineers.

It's worth noting that there is some controversy over whether Hillary and Norgay were the first people to climb Mount Everest. Some people claim t

In [87]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.


In [88]:
!pip install langchain-community

In [89]:
pip install langchain langchain-community langchain-core

Note: you may need to restart the kernel to use updated packages.


In [90]:
import sys
print(sys.executable)

/opt/anaconda3/bin/python


In [91]:
import sys
!{sys.executable} -m pip install -U langchain langchain-community langchain-core

  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.31
    Uninstalling langchain-core-1.2.31:
      Successfully uninstalled langchain-core-1.2.31


In [92]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.accenture.com/us-en/careers/jobdetails?id=R00298545_en&title=SAP+Intercompany+Manager+-+Consumer+Goods")
page_data = loader.load().pop().page_content
print(page_data)






SAP Intercompany Manager - Consumer Goods


























































































































Skip to main content
Skip to footer













Menu




 
Accenture


 


Accenture








Close Menu





What we do

 





Back



What we do





Capabilities
Capabilities



Cloud 
					


Customer Service 
					


Cybersecurity 
					


Data and Artificial Intelligence 
					


Digital Engineering and Manufacturing 
					


Ecosystem Partners 
					


Emerging Technology 
					


Finance and Risk Management 
					


Infrastructure and Capital Projects 
					


Learning 
					


Managed Services 
					


Marketing and Experience 
					


Metaverse 
					


Sales and Commerce 
					


Strategy 
					


Supply Chain 
					


Sustainability 
					


Talent and Organization 
					


Technology Transformation 
					





Industries
Industries



Aerospace and Defense 
					


Automotive 
					


Banking 
					


Capi

In [93]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
    ###SCRAPED TEXT FROM WEBSITE:
    {page_data}
    ### INSTRUCTION:
    The scraped text is from the career's page of a website.
    Your job is to extract the job postings and return them in JSON format containing the 
    following keys: 'role', 'experience', 'skills' and 'description'.
    Only return the valid JSON. 
    ### VALID JSON (NO PREAMBLE):
    """

)
chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
print(res.content)

```json
{
  "role": "SAP Intercompany Manager - Consumer Goods",
  "experience": "Minimum of 9 years SAP functional and technical experience in Intercompany Logistics",
  "skills": [
    "SAP",
    "Intercompany Logistics",
    "Sales",
    "Procurement",
    "Intercompany movements",
    "S/4 implementations",
    "Project planning",
    "Estimation",
    "Solution architecture"
  ],
  "description": "We are looking for a confident leader who spots and stays ahead of the SAP platform, industry and Finance trends and knows how to translate client goals into clear and actionable outcomes. The ideal candidate will have experience managing SAP delivery teams, in a Global Delivery Model, and prior experience in an Advisory and/or Consulting role."
}
```


In [94]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'SAP Intercompany Manager - Consumer Goods',
 'experience': 'Minimum of 9 years SAP functional and technical experience in Intercompany Logistics',
 'skills': ['SAP',
  'Intercompany Logistics',
  'Sales',
  'Procurement',
  'Intercompany movements',
  'S/4 implementations',
  'Project planning',
  'Estimation',
  'Solution architecture'],
 'description': 'We are looking for a confident leader who spots and stays ahead of the SAP platform, industry and Finance trends and knows how to translate client goals into clear and actionable outcomes. The ideal candidate will have experience managing SAP delivery teams, in a Global Delivery Model, and prior experience in an Advisory and/or Consulting role.'}

In [95]:
type(json_res)

dict

In [96]:
import pandas as pd
df= pd.read_csv('Portfolio.csv')
df

,company,employee_name,employee_link,job_title,tech_stack
0,Nike,Arjun Mehta,https://www.linkedin.com/in/arjunmehta,Senior Data Analyst,"Python, SQL, Tableau, Excel"
1,Nike,Emily Johnson,https://www.linkedin.com/in/emilyjohnson,Data Scientist,"Python, Pandas, Scikit-learn, AWS"
2,Nike,Rahul Nair,https://www.linkedin.com/in/rahulnair,BI Analyst,"Power BI, SQL, DAX, Excel"
3,Accenture,Sneha Kapoor,https://www.linkedin.com/in/snehakapoor,SAP Finance Consultant,"SAP S/4HANA, Finance Transformation, AI, Cloud"
4,Accenture,David Lee,https://www.linkedin.com/in/davidlee,AI Engineer,"Python, FastAPI, LLM, Docker, Azure"
5,Accenture,Ankit Sharma,https://www.linkedin.com/in/ankitsharma,Data Engineer,"Python, Spark, Airflow, AWS"
6,Adobe,Michael Brown,https://www.linkedin.com/in/michaelbrown,Machine Learning Engineer,"Python, TensorFlow, NLP, GCP"
7,Adobe,Priya Iyer,https://www.linkedin.com/in/priyaiyer,Data Analyst,"SQL, Excel, Tableau, Statistics"
8,Adobe,Chris Evans,https://www.linkedin.com/in/chrisevans,Backend Engineer,"Node.js, Express, MongoDB, Docker"
9,Google,Ravi Kumar,https://www.linkedin.com/in/ravikumar,Software Engineer,"Java, Spring Boot, Microservices, GCP"


In [97]:
import chromadb
import uuid
client=chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name='Portfolio')

if not collection.count():
    for _, row in df.iterrows():
        collection. add(documents=row['tech_stack'],
                        metadatas={'links':row['employee_link']},
                        ids=[str(uuid.uuid4())])

In [98]:
links= collection.query(query_texts=["Experience in Python, Experience in React"],n_results=2).get('metadatas')
links

[[{'links': 'https://www.linkedin.com/in/lisawong'},
  {'links': 'https://www.linkedin.com/in/annakim'}]]

In [99]:
job=json_res
job['skills']

['SAP',
 'Intercompany Logistics',
 'Sales',
 'Procurement',
 'Intercompany movements',
 'S/4 implementations',
 'Project planning',
 'Estimation',
 'Solution architecture']

In [128]:

prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIOTION:
    {job_description}
    ### INSTRUCTION:
    You are Mohan, a business development executive at BumbleBee. BumbleBee is a consulting company that provides skills and process for seamless integration of business process through automated tools. 
    Over our experience, we have empowered numerous enterprises with tailored solutions. 
    Your job is to write a cold email to the client regarding the job mentioned above fulfiiling all there needs. 
    Also add the most relevant ones from the following links to showcase BumbleBees portfolio: {link_list}
    Remember you are Mohan, BDE at Bumblbee. 
    Do not provide preamble.
    ### Email (NO PREAMBLE):
    """

)
chain_email = prompt_email | llm 
res = chain_email.invoke({'job_description': str(job),'link_list': links})
print(res.content)

Subject: Expert SAP Intercompany Management Solutions for Consumer Goods Industry

Dear Hiring Manager,

I came across your job posting for an SAP Intercompany Manager - Consumer Goods, and I am excited to introduce BumbleBee, a consulting company that specializes in providing tailored solutions for seamless integration of business processes through automated tools. With our expertise, we can empower your enterprise to achieve its goals.

Our team at BumbleBee has a deep understanding of SAP functional and technical requirements, particularly in Intercompany Logistics, Sales, Procurement, and Intercompany movements. We have a proven track record of successfully implementing S/4 solutions, project planning, estimation, and solution architecture. Our experienced professionals have managed SAP delivery teams in a Global Delivery Model and have prior experience in Advisory and Consulting roles.

I'd like to highlight a few examples of our work:

* Our team has worked with industry leaders,